# 🔬 Ablation: Có vs. Không Có `ContextGateValidator` (`32B-AWQ` Outline + Coordinator + LLM Validator | `7B` Chưa Finetune Viết Truyện)

**Phân công model rõ ràng:**
- **Outline (`OutlineAgent`):** `Qwen/Qwen2.5-32B-Instruct-AWQ`
- **Planning & Critic (`Condition D`, không Freytag):** Python CPU (`0` LLM calls)
- **Coordinator (`CoordinatorAgent`):** `Qwen/Qwen2.5-32B-Instruct-AWQ`
- **Validator (`ContextGateValidator`):**
  - `no_validator`: Bỏ hẳn Validator (dùng thẳng JSON từ Coordinator `32B-AWQ`, chỉ fallback nếu vỡ JSON)
  - `with_validator_heuristic`: Python Heuristic (`threshold = 0.80`)
  - `with_validator_llm`: **`Qwen/Qwen2.5-32B-Instruct-AWQ`** (`temperature = 0.0`, `threshold = 0.80`)
- **Writing Agent (chưa finetune):** `Qwen/Qwen2.5-7B-Instruct`

**Tối ưu VRAM tự động cho cả A100 40GB lẫn A100 80GB:**
- Trên **A100 40GB**: chạy **đồng bộ theo từng chương (`ch = 1 → 8`)** gom batch toàn bộ `(arm × 3 truyện)` vào mỗi lượt `7B` và `32B-AWQ` → **0 OOM**, chỉ đổi model đúng `7` lượt cho toàn bộ thí nghiệm!
- Trên **A100 80GB**: giữ song song cả `32B-AWQ` và `7B` trong VRAM (`0` lần đổi model).
- **Chưa chấm metric vội:** lưu toàn bộ văn bản 7B sinh ra + log chuyển chương sang Google Drive (`/content/drive/MyDrive/MA-GRPO-ContextGate-Ablation/`).

---

## Bước 1: Kiểm tra GPU, Mount Google Drive & Clone `MA-GRPO`

In [ ]:
!nvidia-smi

import os
import sys
import subprocess
import getpass
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_DIR = Path("/content/drive/MyDrive/MA-GRPO-ContextGate-Ablation")
except Exception:
    OUTPUT_DIR = Path("ablation_context_gate_outputs")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Thư mục lưu kết quả trên Drive: {OUTPUT_DIR}")

REPO_URL = "github.com/AIVIETNAM-AIO-TruongDogki/MA_GRPO.git"
REPO_DIR = "/content/MA-GRPO"

if os.path.exists("/content"):
    try:
        from google.colab import userdata
        token = userdata.get('GH_TOKEN')
    except Exception:
        token = None

    if not os.path.exists(REPO_DIR):
        if token:
            subprocess.run(["git", "clone", f"https://{token}@{REPO_URL}", REPO_DIR], check=True)
        else:
            res = subprocess.run(["git", "clone", f"https://{REPO_URL}", REPO_DIR])
            if res.returncode != 0:
                token = getpass.getpass("Nhập GitHub Personal Access Token: ")
                subprocess.run(["git", "clone", f"https://{token}@{REPO_URL}", REPO_DIR], check=True)
    else:
        subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)
    os.chdir(REPO_DIR)
    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)

assert os.path.exists("src/coordinator.py"), "Không tìm thấy src/coordinator.py!"
print("Working dir:", os.getcwd())
!git log -n 1 --oneline


## Bước 2: Cài đặt `vllm` & Thư viện cần thiết

In [ ]:
!pip install -q vllm pandas pyyaml pydantic
print("✅ Cài đặt hoàn tất!")


## Bước 3 (Pha 1 — `Qwen/Qwen2.5-32B-Instruct-AWQ`): Sinh Macro Outline & Planning Chung Cho 3 Đề Bài MoPS
- Sinh **1 bộ Macro Plan (`Condition D`, không Freytag)** cố định dùng chung cho tất cả các nhánh.
- Lưu `shared_macro_plans.json` lên Drive để lần sau có thể tái sử dụng ngay (`REUSE_SAVED_PLANS = True`).

In [ ]:
import gc
import json
import time
import torch
from vllm import LLM, SamplingParams
from src.dataset_generator import load_premises_pool
from src.planning import OutlineAgent, PlanCriticAgent, PlanningAgent
from src.pipeline import resolve_optimal_dtype

# --- CẤU HÌNH CHUNG ---
PLANNER_COORD_MODEL_ID = "Qwen/Qwen2.5-32B-Instruct-AWQ"   # Outline + Coordinator + LLM Validator
WRITER_MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"               # 7B chưa finetune chuyên viết truyện
NUM_STORIES = 3
NUM_CHAPTERS = 8
TARGET_WORDS_CHOICES = (8000, 9000, 10000)
REUSE_SAVED_PLANS = True

VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0.0
CO_RESIDENT_80GB = VRAM_GB >= 70.0
print(f"GPU VRAM: {VRAM_GB:.1f} GB -> Chế độ: {'Giữ song song 32B + 7B (80GB)' if CO_RESIDENT_80GB else 'Swap gọn theo từng chương (40GB, 0 OOM)'}")

def load_vllm_engine(model_id: str, max_model_len: int = 12288, gpu_mem_util: float = 0.88):
    lower_m = model_id.lower()
    dtype = "float16" if ("awq" in lower_m or "gptq" in lower_m) else resolve_optimal_dtype()
    print(f"[vLLM] Loading {model_id} (dtype={dtype}, max_len={max_model_len}, mem={gpu_mem_util})...", flush=True)
    eng = LLM(
        model=model_id,
        dtype=dtype,
        gpu_memory_utilization=gpu_mem_util,
        max_model_len=max_model_len,
        trust_remote_code=True,
    )
    return eng, eng.get_tokenizer(), max_model_len

def unload_vllm_engine(eng, tok):
    del eng, tok
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def batch_chat(eng, tok, max_model_len: int, requests, max_tokens: int, temperature: float, seed: int = 42):
    if not requests:
        return []
    formatted = []
    max_p_tok = 0
    for sys_p, usr_p in requests:
        txt = tok.apply_chat_template(
            [{"role": "system", "content": sys_p}, {"role": "user", "content": usr_p}],
            tokenize=False,
            add_generation_prompt=True,
        )
        formatted.append(txt)
        max_p_tok = max(max_p_tok, len(tok.encode(txt, add_special_tokens=False)))
    safe_max = max(64, min(max_tokens, max_model_len - max_p_tok - 32))
    params = SamplingParams(
        temperature=temperature,
        top_p=0.9,
        max_tokens=safe_max,
        presence_penalty=0.1,
        seed=seed,
    )
    outputs = eng.generate(formatted, params, use_tqdm=False)
    return [o.outputs[0].text.strip() if o.outputs else "" for o in outputs]

plans_file = OUTPUT_DIR / "shared_macro_plans.json"

if REUSE_SAVED_PLANS and plans_file.exists():
    shared_plans = json.loads(plans_file.read_text(encoding="utf-8"))
    print(f"✅ Đã nạp lại {len(shared_plans)} Macro Plans từ {plans_file}")
else:
    premises, excluded = load_premises_pool(
        premises_files=["data/mops_premises.jsonl"],
        include_canonical=False,
        exclude_files=["data/tell-me-a-story-test.jsonl"],
        num_premises=NUM_STORIES,
        target_words_choices=TARGET_WORDS_CHOICES,
        num_chapters=NUM_CHAPTERS,
    )
    print(f"Đã chọn {len(premises)} đề bài MoPS (loại {excluded} trùng tập test).", flush=True)

    eng32, tok32, len32 = load_vllm_engine(PLANNER_COORD_MODEL_ID, max_model_len=12288, gpu_mem_util=0.88)
    try:
        outline_agent = OutlineAgent(
            llm_generate_fn=lambda s, u: batch_chat(eng32, tok32, len32, [(s, u)], max_tokens=2048, temperature=0.7)[0]
        )
        planning_agent = PlanningAgent(sub_event_agent=None, critic=PlanCriticAgent(), enable_micro_subevents=False)

        shared_plans = []
        for idx, item in enumerate(premises, 1):
            t0 = time.time()
            print(f"  -> [{idx}/{len(premises)}] Outline (32B) + Planning: {item['id']} ({item['genre']}, {item['target_words']}w)", flush=True)
            eg = outline_agent.create_event_graph(item["premise"], genre=item["genre"], num_events=NUM_CHAPTERS)
            sp = planning_agent.partition_into_chapters(
                event_graph=eg,
                total_target_words=item["target_words"],
                num_chapters=NUM_CHAPTERS,
                enable_micro_subevents=False,
            )
            shared_plans.append({"item": item, "event_graph": eg, "story_plan": sp})
            print(f"     Xong trong {time.time() - t0:.1f}s | planning_tokens={eg.get('planning_tokens')}", flush=True)
    finally:
        unload_vllm_engine(eng32, tok32)

    plans_file.write_text(json.dumps(shared_plans, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"\n✅ Đã lưu Macro Plans chung tại: {plans_file} và xả VRAM 32B!")


## Bước 4 (Pha 2 — Vòng Lặp Đồng Bộ Theo Chương `ch = 1 → 8`): `7B` Viết Truyện + `32B-AWQ` Coordinator & Validator
- **Mỗi lượt chương `k`:**
  1. `Qwen/Qwen2.5-7B-Instruct` (chưa finetune) viết song song Chương `k` cho toàn bộ `(arm × 3 truyện)` trong **1 batch vLLM duy nhất**.
  2. Nếu `k < 8`: `Qwen/Qwen2.5-32B-Instruct-AWQ` chạy **`CoordinatorAgent`** song song cho toàn bộ `(arm × 3 truyện)` trong **1 batch vLLM**, sau đó chạy tiếp **Validator tương ứng từng nhánh** (`no_validator` bỏ qua hoàn toàn | `with_validator_heuristic` chấm CPU | `with_validator_llm` chấm bằng `32B-AWQ` ở `temp=0.0`).
- Toàn bộ văn bản 7B sinh ra + log chuyển chương được lưu sạch sang Google Drive.

In [ ]:
import json
import time
from src.coordinator import (
    COORDINATOR_TWO_TIER_PROMPT,
    VALIDATOR_MULTI_SIGNAL_PROMPT,
    ContextGateValidator,
    CoordinatorAgent,
    clean_json_response,
    detect_heuristic_negation_contradictions,
)
from src.pipeline import VLLMWritingPolicy, extract_cot_and_prose

RUN_ARMS = [
    "no_validator",
    "with_validator_heuristic",
    "with_validator_llm",
]
WRITER_SEED = 42
WRITER_MAX_LEN = 16384
COORD_MAX_LEN = 12288

prompt_builder = VLLMWritingPolicy(use_mock=True)
helper_coord = CoordinatorAgent(llm_generate_fn=None, validator=ContextGateValidator(None, threshold=0.80))
heuristic_validator = ContextGateValidator(llm_generate_fn=None, threshold=0.80)

# Khởi tạo trạng thái cho tất cả các cặp (arm, story)
states = []
for arm in RUN_ARMS:
    for bundle in shared_plans:
        item = bundle["item"]
        sp = bundle["story_plan"]
        init_chars = sp["chapters"][0].get("characters_active", ["Protagonist"])
        init_ctx = helper_coord.condense_context(1, [], [], init_chars)
        states.append({
            "arm": arm,
            "item": item,
            "story_plan": sp,
            "ctx": init_ctx,
            "past_chapters": [],
            "past_raw_outputs": [],
            "past_plans": [],
            "transitions": [],
            "fallback_count": 0,
            "coord_latency_s": 0.0,
            "writer_latency_s": 0.0,
        })

eng7 = tok7 = len7 = None
eng32 = tok32 = len32 = None

if CO_RESIDENT_80GB:
    eng32, tok32, len32 = load_vllm_engine(PLANNER_COORD_MODEL_ID, max_model_len=COORD_MAX_LEN, gpu_mem_util=0.45)
    eng7, tok7, len7 = load_vllm_engine(WRITER_MODEL_ID, max_model_len=WRITER_MAX_LEN, gpu_mem_util=0.35)

try:
    for ch_idx in range(1, NUM_CHAPTERS + 1):
        print(f"\n==================== CHAPTER {ch_idx}/{NUM_CHAPTERS} (All {len(states)} runs in parallel) ====================", flush=True)

        # -----------------------------------------------------------------
        # STEP A: Qwen/Qwen2.5-7B-Instruct (chưa finetune) viết Chương ch_idx
        # -----------------------------------------------------------------
        if not CO_RESIDENT_80GB:
            eng7, tok7, len7 = load_vllm_engine(WRITER_MODEL_ID, max_model_len=WRITER_MAX_LEN, gpu_mem_util=0.85)

        t_w = time.time()
        write_reqs = []
        for st in states:
            ch_plan = st["story_plan"]["chapters"][ch_idx - 1]
            w_prompt = prompt_builder.construct_prompt(ch_plan, st["ctx"], enable_cot=True)
            write_reqs.append(("You are an elite literary novelist.", w_prompt))

        raw_chapters = batch_chat(
            eng7, tok7, len7, write_reqs,
            max_tokens=WRITER_MAX_LEN,
            temperature=0.7,
            seed=WRITER_SEED + ch_idx,
        )
        w_elapsed = time.time() - t_w
        for st, raw_out in zip(states, raw_chapters):
            ch_plan = st["story_plan"]["chapters"][ch_idx - 1]
            _, prose = extract_cot_and_prose(raw_out)
            prose = prose if prose else raw_out
            st["past_raw_outputs"].append(raw_out)
            st["past_chapters"].append(prose)
            st["past_plans"].append(ch_plan)
            st["writer_latency_s"] += w_elapsed / len(states)

        avg_w = sum(len(st["past_chapters"][-1].split()) for st in states) / len(states)
        print(f"  [7B Writer] Xong Ch {ch_idx} cho {len(states)} runs trong {w_elapsed:.1f}s (TB {avg_w:.0f} từ/chương)", flush=True)

        if not CO_RESIDENT_80GB:
            unload_vllm_engine(eng7, tok7)
            eng7 = tok7 = None

        # Nếu là chương cuối thì không cần chạy Coordinator nữa
        if ch_idx == NUM_CHAPTERS:
            break

        # -----------------------------------------------------------------
        # STEP B: Qwen/Qwen2.5-32B-Instruct-AWQ chạy Coordinator + Validator
        # -----------------------------------------------------------------
        if not CO_RESIDENT_80GB:
            eng32, tok32, len32 = load_vllm_engine(PLANNER_COORD_MODEL_ID, max_model_len=COORD_MAX_LEN, gpu_mem_util=0.88)

        t_c = time.time()
        coord_reqs = []
        tails = []
        for st in states:
            prev_text = st["past_chapters"][-1]
            tail = " ".join(prev_text.strip().split()[-180:])
            tails.append(tail)
            last_p = st["past_plans"][-1]
            u_prompt = (
                f"Total chapters completed: {len(st['past_chapters'])}.\n"
                f"Upcoming chapter index: {ch_idx + 1}.\n\n"
                f"[LAST CHAPTER PROSE (TAIL)]:\n{prev_text[-2500:]}\n\n"
                f"[LAST CHAPTER PLAN GOAL]:\n{last_p.get('chapter_goal', '')}\n"
                f"[LAST CHAPTER EXIT STATE]:\n{last_p.get('exit_state', '')}\n\n"
                "Synthesize Two-Tier Context State matching the JSON schema.\n"
            )
            coord_reqs.append((COORDINATOR_TWO_TIER_PROMPT, u_prompt))

        coord_raws = batch_chat(
            eng32, tok32, len32, coord_reqs,
            max_tokens=768,
            temperature=0.3,
            seed=WRITER_SEED + 100 + ch_idx,
        )

        # Parse ứng viên tóm tắt từ 32B-AWQ Coordinator
        candidates = []
        json_fb_flags = []
        for st, tail, raw_c in zip(states, tails, coord_raws):
            next_p = st["story_plan"]["chapters"][ch_idx]
            chars = next_p.get("characters_active", ["Protagonist"])
            cand = None
            try:
                data = json.loads(clean_json_response(raw_c))
                long_m = data.get("long_term_memory", {})
                short_m = data.get("short_term_memory", {})
                if not isinstance(long_m, dict) or not isinstance(short_m, dict):
                    raise ValueError("Invalid JSON shape")
                short_m["recent_chapter_index"] = len(st["past_chapters"])
                short_m["recent_tail_excerpt"] = tail
                plot_sum = str(short_m.get("recent_chapter_summary", "")).strip()
                if not plot_sum:
                    raise ValueError("Empty summary")
                c_map = long_m.get("character_states", {})
                c_list = [f"{k}: {v}" for k, v in c_map.items()] if isinstance(c_map, dict) and c_map else [f"{c}: Active" for c in chars]
                cand = {
                    "chapter_index": ch_idx + 1,
                    "completed_chapters": len(st["past_chapters"]),
                    "long_term_memory": long_m,
                    "short_term_memory": short_m,
                    "plot_summary": plot_sum,
                    "character_statuses": c_list,
                    "pending_threads": long_m.get("unresolved_plot_threads", []),
                    "recent_tail_excerpt": tail,
                }
                json_fb_flags.append(False)
            except Exception:
                cand = helper_coord._structured_state_extraction(
                    ch_idx + 1, st["past_chapters"], st["past_plans"], chars, tail, st["ctx"], next_p
                )
                json_fb_flags.append(True)
            candidates.append(cand)

        # Nếu có nhánh with_validator_llm -> gom batch gọi 32B-AWQ chấm 1 lần duy nhất
        llm_val_indices = [i for i, st in enumerate(states) if st["arm"] == "with_validator_llm"]
        llm_val_raw_map = {}
        if llm_val_indices:
            val_reqs = []
            for i in llm_val_indices:
                st = states[i]
                cand = candidates[i]
                prev_text = st["past_chapters"][-1]
                prev_goal = st["past_plans"][-1].get("chapter_goal", "")
                next_goal = st["story_plan"]["chapters"][ch_idx].get("chapter_goal", "")
                lm = cand.get("long_term_memory", {})
                v_user = (
                    f"[PRECEDING CHAPTER PROSE EXCERPTS (Source of Truth)]:\n"
                    f"--- BEGINNING OF CHAPTER ---\n{prev_text[:1200]}\n...\n"
                    f"--- CONCLUSION OF CHAPTER ---\n{prev_text[-1800:]}\n\n"
                    f"[PRECEDING CHAPTER GOAL]: {prev_goal}\n"
                    f"[UPCOMING CHAPTER GOAL]: {next_goal}\n\n"
                    f"[PROPOSED CANDIDATE CONTEXT]:\n"
                    f"Short-Term Recent Summary: {cand.get('plot_summary', '')}\n"
                    f"Character States: {lm.get('character_states', cand.get('character_statuses', []))}\n"
                    f"Unresolved Threads: {lm.get('unresolved_plot_threads', cand.get('pending_threads', []))}\n"
                    f"Major Past Events: {lm.get('major_events', [])}\n\n"
                    "Audit faithfulness (0.0-1.0) and completeness (0.0-1.0).\n"
                )
                val_reqs.append((VALIDATOR_MULTI_SIGNAL_PROMPT, v_user))
            val_outs = batch_chat(
                eng32, tok32, len32, val_reqs,
                max_tokens=512,
                temperature=0.0,
                seed=WRITER_SEED + 200 + ch_idx,
            )
            for i, v_raw in zip(llm_val_indices, val_outs):
                llm_val_raw_map[i] = v_raw

        c_elapsed = time.time() - t_c
        if not CO_RESIDENT_80GB:
            unload_vllm_engine(eng32, tok32)
            eng32 = tok32 = None

        # Áp dụng quyết định Validator cho từng nhánh
        for i, st in enumerate(states):
            arm = st["arm"]
            cand = candidates[i]
            tail = tails[i]
            prev_text = st["past_chapters"][-1]
            next_p = st["story_plan"]["chapters"][ch_idx]
            chars = next_p.get("characters_active", ["Protagonist"])

            if arm == "no_validator":
                final_ctx = cand
                is_fb = json_fb_flags[i]
                val_meta = {"bypassed": True, "is_valid": True, "is_fallback": is_fb, "composite_score": None, "issues": []}
            elif arm == "with_validator_heuristic":
                val_meta = heuristic_validator.validate_summary(
                    summary=cand, prev_chapter_text=prev_text, prev_chapter_plan=st["past_plans"][-1],
                    previous_context=st["ctx"], next_chapter_plan=next_p
                )
                val_meta["bypassed"] = False
                if val_meta.get("is_valid", False):
                    final_ctx = cand
                    is_fb = json_fb_flags[i]
                else:
                    final_ctx = helper_coord._structured_state_extraction(
                        ch_idx + 1, st["past_chapters"], st["past_plans"], chars, tail, st["ctx"], next_p
                    )
                    is_fb = True
            elif arm == "with_validator_llm":
                neg_clashes = detect_heuristic_negation_contradictions(cand.get("plot_summary", ""), prev_text)
                issues = list(neg_clashes)
                try:
                    vdata = json.loads(clean_json_response(llm_val_raw_map[i]))
                    faith = float(vdata.get("faithfulness_score", 0.85))
                    comp = float(vdata.get("completeness_score", 0.85))
                    llm_contra = vdata.get("negation_contradictions", [])
                    feedback = str(vdata.get("feedback", "LLM 32B validation processed."))
                    if llm_contra:
                        issues.extend(llm_contra)
                    if neg_clashes:
                        faith = max(0.0, faith - (0.25 * len(set(neg_clashes) | set(llm_contra)) if llm_contra else 0.05 * len(neg_clashes)))
                except Exception:
                    faith, comp = heuristic_validator._heuristic_evaluation(cand, prev_text, next_p)
                    feedback = "Fallback heuristic scoring applied."
                comp_score = round(0.5 * faith + 0.5 * comp, 3)
                is_valid = comp_score >= 0.80
                val_meta = {
                    "bypassed": False,
                    "faithfulness_score": round(faith, 3),
                    "completeness_score": round(comp, 3),
                    "composite_score": comp_score,
                    "is_valid": is_valid,
                    "is_fallback": not is_valid,
                    "feedback": feedback,
                    "issues": issues,
                }
                if is_valid:
                    final_ctx = cand
                    is_fb = json_fb_flags[i]
                else:
                    final_ctx = helper_coord._structured_state_extraction(
                        ch_idx + 1, st["past_chapters"], st["past_plans"], chars, tail, st["ctx"], next_p
                    )
                    is_fb = True
            else:
                raise ValueError(arm)

            final_ctx["validation_metadata"] = val_meta
            if is_fb:
                st["fallback_count"] += 1
            st["coord_latency_s"] += c_elapsed / len(states)
            st["transitions"].append({
                "arm": arm,
                "premise_id": st["item"]["id"],
                "chapter_index": ch_idx + 1,
                "candidate_summary": cand.get("plot_summary", ""),
                "final_summary": final_ctx.get("plot_summary", ""),
                "prev_chapter_tail": tail,
                "context_state": final_ctx,
                "validation_metadata": val_meta,
                "is_fallback": is_fb,
            })
            st["ctx"] = final_ctx

        fb_str = ", ".join(f"{arm}={sum(1 for s in states if s['arm']==arm and s['transitions'][-1]['is_fallback'])}" for arm in RUN_ARMS)
        print(f"  [32B Coordinator + Validator] Xong Ch {ch_idx}->Ch {ch_idx+1} trong {c_elapsed:.1f}s | fallbacks: {fb_str}", flush=True)
finally:
    if eng7 is not None:
        unload_vllm_engine(eng7, tok7)
    if eng32 is not None:
        unload_vllm_engine(eng32, tok32)

# Lưu toàn bộ sản phẩm ra Google Drive
stories_jsonl = OUTPUT_DIR / "stories_7b_outputs.jsonl"
transitions_jsonl = OUTPUT_DIR / "coordinator_transitions.jsonl"
stories_dir = OUTPUT_DIR / "stories"
stories_dir.mkdir(parents=True, exist_ok=True)

all_story_rows = []
with open(stories_jsonl, "w", encoding="utf-8") as fs, open(transitions_jsonl, "w", encoding="utf-8") as ft:
    for st in states:
        item = st["item"]
        sp = st["story_plan"]
        full_story = "\n\n".join(st["past_chapters"])
        actual_words = len(full_story.split())
        rec = {
            "arm": st["arm"],
            "planner_model": PLANNER_COORD_MODEL_ID,
            "coordinator_model": PLANNER_COORD_MODEL_ID,
            "validator_model": PLANNER_COORD_MODEL_ID if st["arm"] == "with_validator_llm" else ("heuristic" if st["arm"] == "with_validator_heuristic" else None),
            "writer_model": WRITER_MODEL_ID,
            "premise_id": item["id"],
            "genre": item["genre"],
            "premise": item["premise"],
            "target_words": item["target_words"],
            "num_chapters": len(st["past_chapters"]),
            "actual_words": actual_words,
            "chapter_word_counts": [len(c.split()) for c in st["past_chapters"]],
            "fallback_count": st["fallback_count"],
            "n_transitions": len(st["transitions"]),
            "coord_latency_s": round(st["coord_latency_s"], 2),
            "writer_latency_s": round(st["writer_latency_s"], 2),
            "chapters": st["past_chapters"],
            "chapters_raw_with_think": st["past_raw_outputs"],
            "full_story": full_story,
            "story_plan": sp,
        }
        all_story_rows.append(rec)
        fs.write(json.dumps(rec, ensure_ascii=False) + "\n")
        for tr in st["transitions"]:
            ft.write(json.dumps(tr, ensure_ascii=False) + "\n")

        arm_dir = stories_dir / st["arm"]
        arm_dir.mkdir(parents=True, exist_ok=True)
        md_parts = [
            f"# [{st['arm']}] Story `{item['id']}` ({item['genre']})",
            f"- **Outline + Coordinator:** `{PLANNER_COORD_MODEL_ID}` | **Writer:** `{WRITER_MODEL_ID}` (untrained)",
            f"- **Premise:** {item['premise']}",
            f"- **Target Words:** {item['target_words']} | **Actual Words (7B):** {actual_words} | **Validator Fallbacks:** {st['fallback_count']}/{len(st['transitions'])}\n",
        ]
        for ch_p, ch_txt in zip(sp["chapters"], st["past_chapters"]):
            md_parts.append(f"## {ch_p['chapter_title']} ({len(ch_txt.split())} words)\n")
            md_parts.append(ch_txt + "\n")
        (arm_dir / f"{item['id']}.md").write_text("\n".join(md_parts), encoding="utf-8")

print(f"\n✅ Đã lưu toàn bộ {len(all_story_rows)} truyện 7B vào Google Drive: {OUTPUT_DIR}")


## Bước 5: Thống Kê Nhanh Các File Đã Lưu Trên Google Drive
*(Chưa chấm metric — toàn bộ văn bản 7B sinh ra & log chuyển chương đã lưu sạch trong Drive để thiết lập metric đánh giá sau.)*

In [ ]:
import pandas as pd

summary = [
    {
        "arm": r["arm"],
        "premise_id": r["premise_id"][:14] + "...",
        "genre": r["genre"],
        "target_w": r["target_words"],
        "actual_w_7b": r["actual_words"],
        "fallbacks": f"{r['fallback_count']}/{r['n_transitions']}",
        "writer_s": r["writer_latency_s"],
        "coord_s": r["coord_latency_s"],
    }
    for r in all_story_rows
]
print("=== TÓM TẮT KẾT QUẢ SINH TRUYỆN 7B (ĐÃ LƯU GOOGLE DRIVE) ===")
print(pd.DataFrame(summary).to_string(index=False))

print("\n=== CÁC FILE ĐÃ LƯU TRÊN GOOGLE DRIVE ===")
for p in sorted(OUTPUT_DIR.rglob("*")):
    if p.is_file():
        print(f"  - {p} ({p.stat().st_size / 1024:.1f} KB)")


## Bước 6: Tự động Ngắt Server Colab Sau Khi Lưu Xong Lên Drive

In [ ]:
import time

AUTO_DISCONNECT = True

if AUTO_DISCONNECT:
    print("✅ Toàn bộ truyện 7B đã lưu vào Google Drive! Chờ 15 giây đồng bộ rồi tự ngắt runtime...")
    time.sleep(15)
    try:
        from google.colab import runtime
        runtime.unassign()
    except Exception as e:
        print(f"Không thể gọi runtime.unassign(): {e}")
else:
    print("ℹ️ AUTO_DISCONNECT=False — Nhớ tắt runtime thủ công sau khi kiểm tra.")
